# ML-Driven DCF Valuation Engine -- Part 3: Results & Signals

```
FMP Financial Data          (Part 1)
        |
Merge Financial Statements  (Part 1)
        |
Feature Engineering         (Part 1)
        |
Train ML Models             (Part 2)
        |
Forecast & DCF Valuation    (Part 2)
        |
Compare vs Market Price     <-- YOU ARE HERE
        |
Buy / Hold / Sell Signal    <-- YOU ARE HERE
```

---
**Prerequisites:** Run Parts 1 & 2 first, or run the setup cell below to load, train, and value everything.

### Setup -- Re-run Full Pipeline from Parts 1 & 2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# -- Load CSVs --
income     = pd.read_csv('fmp_dataset/csv/income-statement.csv')
balance    = pd.read_csv('fmp_dataset/csv/balance-sheet-statement.csv')
cashflow   = pd.read_csv('fmp_dataset/csv/cash-flow-statement.csv')
enterprise = pd.read_csv('fmp_dataset/csv/enterprise-values.csv')
quote      = pd.read_csv('fmp_dataset/csv/quote.csv')
dcf_fmp    = pd.read_csv('fmp_dataset/csv/discounted-cash-flow.csv')

# -- Select columns & merge --
inc_cols = ['date','symbol','revenue','costOfRevenue','grossProfit',
    'researchAndDevelopmentExpenses','sellingGeneralAndAdministrativeExpenses',
    'operatingExpenses','operatingIncome','ebit','ebitda',
    'depreciationAndAmortization','interestExpense',
    'incomeBeforeTax','incomeTaxExpense','netIncome',
    'eps','epsDiluted','weightedAverageShsOut','weightedAverageShsOutDil']
bs_cols = ['date','symbol','cashAndCashEquivalents','shortTermInvestments',
    'netReceivables','inventory','totalCurrentAssets',
    'propertyPlantEquipmentNet','goodwillAndIntangibleAssets',
    'totalAssets','accountPayables','totalCurrentLiabilities',
    'longTermDebt','totalLiabilities','totalStockholdersEquity','totalDebt','netDebt']
cf_cols = ['date','symbol','stockBasedCompensation','changeInWorkingCapital',
    'operatingCashFlow','capitalExpenditure','freeCashFlow',
    'acquisitionsNet','netCashProvidedByOperatingActivities',
    'commonStockRepurchased','commonDividendsPaid']
ev_cols = ['date','symbol','stockPrice','numberOfShares','marketCapitalization','enterpriseValue']

merged = income[inc_cols].merge(balance[bs_cols], on=['symbol','date'], how='inner')
merged = merged.merge(cashflow[cf_cols], on=['symbol','date'], how='inner')
merged = merged.merge(enterprise[ev_cols], on=['symbol','date'], how='inner')
merged['date'] = pd.to_datetime(merged['date'])
merged = merged.sort_values(['symbol','date']).reset_index(drop=True)

# -- Feature engineering --
def engineer_features(group):
    g = group.copy()
    rev = g['revenue']
    g['rev_growth'] = rev.pct_change()
    g['gross_margin']  = g['grossProfit'] / rev
    g['ebit_margin']   = g['ebit'] / rev
    g['eff_tax_rate'] = (g['incomeTaxExpense'] / g['incomeBeforeTax']).clip(0, 0.40)
    g['da_pct_rev']    = g['depreciationAndAmortization'] / rev
    g['capex_pct_rev'] = g['capitalExpenditure'].abs() / rev
    g['nwc']           = g['totalCurrentAssets'] - g['totalCurrentLiabilities']
    g['nwc_pct_rev']   = g['nwc'] / rev
    g['delta_nwc']     = g['nwc'].diff()
    driver_cols = ['rev_growth','gross_margin','ebit_margin',
                   'eff_tax_rate','da_pct_rev','capex_pct_rev','nwc_pct_rev']
    for col in driver_cols:
        g[f'{col}_lag1'] = g[col].shift(1)
    return g

merged = merged.groupby('symbol', group_keys=False).apply(engineer_features)
merged = merged.replace([np.inf, -np.inf], np.nan)

# -- Train models --
FEATURES = ['rev_growth_lag1','gross_margin_lag1','ebit_margin_lag1',
    'eff_tax_rate_lag1','da_pct_rev_lag1','capex_pct_rev_lag1','nwc_pct_rev_lag1']
TARGETS = ['rev_growth','gross_margin','ebit_margin',
    'eff_tax_rate','da_pct_rev','capex_pct_rev','nwc_pct_rev']

train_df = merged.dropna(subset=FEATURES + TARGETS).copy()
models = {}
for target in TARGETS:
    rf = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=42)
    rf.fit(train_df[FEATURES], train_df[target])
    models[target] = rf

# -- DCF valuation --
WACC, TERMINAL_GROWTH, FORECAST_YEARS = 0.09, 0.025, 5
latest = merged.sort_values('date').groupby('symbol').tail(1).copy()
latest = latest.dropna(subset=FEATURES)

all_results = []
for _, row in latest.iterrows():
    sym, curr_rev, curr_nwc = row['symbol'], row['revenue'], row['nwc']
    feat_dict = {f: row[f] for f in FEATURES}
    feat_input = pd.DataFrame([feat_dict])
    fcff_list = []
    for yr in range(1, FORECAST_YEARS + 1):
        preds = {t: models[t].predict(feat_input)[0] for t in TARGETS}
        proj_rev   = curr_rev * (1 + preds['rev_growth'])
        proj_ebit  = proj_rev * preds['ebit_margin']
        proj_nopat = proj_ebit - proj_ebit * preds['eff_tax_rate']
        proj_da    = proj_rev * preds['da_pct_rev']
        proj_capex = proj_rev * preds['capex_pct_rev']
        proj_nwc   = proj_rev * preds['nwc_pct_rev']
        fcff = proj_nopat + proj_da - proj_capex - (proj_nwc - curr_nwc)
        fcff_list.append(fcff)
        feat_dict = {f'{t}_lag1': preds[t] for t in TARGETS}
        feat_input = pd.DataFrame([feat_dict])
        curr_rev, curr_nwc = proj_rev, proj_nwc
    disc = [(1 + WACC) ** yr for yr in range(1, FORECAST_YEARS + 1)]
    pv_fcff = sum(f / d for f, d in zip(fcff_list, disc))
    tv = (fcff_list[-1] * (1 + TERMINAL_GROWTH)) / (WACC - TERMINAL_GROWTH)
    pv_tv = tv / disc[-1]
    ev = pv_fcff + pv_tv
    eq = ev + row['cashAndCashEquivalents'] - row['totalDebt']
    shares = row['weightedAverageShsOutDil']
    all_results.append({
        'Symbol': sym, 'Last FY Revenue': row['revenue'],
        'PV of FCFFs': pv_fcff, 'PV of Terminal Value': pv_tv,
        'Enterprise Value': ev, '(+) Cash': row['cashAndCashEquivalents'],
        '(-) Debt': row['totalDebt'], 'Equity Value': eq,
        'Shares Outstanding': shares,
        'ML Intrinsic Value': eq / shares if shares > 0 else np.nan
    })

valuation = pd.DataFrame(all_results)
print(f'[Setup] Pipeline complete: {len(valuation)} companies valued.')

---
## Steps 9 & 10 -- Compare vs Market Price -> Buy / Hold / Sell Signal
Merge with live quote prices, compute the margin of safety, and generate a trading signal.

| Signal | Condition |
|--------|----------|
| **BUY** | Intrinsic Value > 20% above Market Price (undervalued) |
| **HOLD** | Within +/-20% band |
| **SELL** | Intrinsic Value > 20% below Market Price (overvalued) |

In [ ]:
# -- Bring in live market prices from the quote CSV --
live_prices = quote[['symbol', 'name', 'price']].rename(
    columns={'symbol': 'Symbol', 'price': 'Market Price', 'name': 'Company'}
)

# -- Also bring in FMP's own DCF for comparison --
fmp_dcf = dcf_fmp[['symbol', 'dcf']].rename(
    columns={'symbol': 'Symbol', 'dcf': 'FMP DCF Value'}
)

# -- Merge everything --
final = valuation[['Symbol', 'ML Intrinsic Value']].merge(live_prices, on='Symbol', how='inner')
final = final.merge(fmp_dcf, on='Symbol', how='left')

# -- Compute upside / downside --
final['ML Upside %'] = ((final['ML Intrinsic Value'] - final['Market Price']) / final['Market Price']) * 100

# -- Generate Signal --
def signal(upside):
    if pd.isna(upside):
        return 'N/A'
    if upside > 20:
        return '[+] BUY'
    elif upside < -20:
        return '[-] SELL'
    else:
        return '[=] HOLD'

final['Signal'] = final['ML Upside %'].apply(signal)

# -- Format for display --
display_df = final[['Symbol', 'Company', 'Market Price',
                     'ML Intrinsic Value', 'FMP DCF Value',
                     'ML Upside %', 'Signal']].copy()

display_df['Market Price']       = display_df['Market Price'].apply(lambda x: f'${x:,.2f}')
display_df['ML Intrinsic Value'] = display_df['ML Intrinsic Value'].apply(lambda x: f'${x:,.2f}')
display_df['FMP DCF Value']      = display_df['FMP DCF Value'].apply(lambda x: f'${x:,.2f}')
display_df['ML Upside %']        = display_df['ML Upside %'].apply(lambda x: f'{x:+.1f}%')

display_df = display_df.sort_values('Signal')
print('\n===  FINAL VALUATION REPORT  ===\n')
display_df

---
## Appendix -- DCF Bridge (Detailed Breakdown)
Show the full Enterprise Value -> Equity Value waterfall for every company.

In [ ]:
# -- Full DCF waterfall for each company --
bridge = valuation.copy()
money_bridge_cols = ['Last FY Revenue', 'PV of FCFFs', 'PV of Terminal Value',
                     'Enterprise Value', '(+) Cash', '(-) Debt', 'Equity Value']

for col in money_bridge_cols:
    bridge[col] = bridge[col].apply(lambda x: f'${x/1e9:,.1f}B')

bridge['Shares Outstanding'] = bridge['Shares Outstanding'].apply(lambda x: f'{x/1e9:,.2f}B')
bridge['ML Intrinsic Value'] = bridge['ML Intrinsic Value'].apply(lambda x: f'${x:,.2f}')

print('\n-- DCF Bridge: Enterprise Value -> Intrinsic Value per Share --\n')
bridge